# Individuell uppgift – Hälsostudie

## Introduktion

Du arbetar som dataanalytiker på ett forskningsinstitut och har fått tillgång till data från en hälsostudie.

Datasetet innehåller information om deltagarnas ålder, kön, vikt, längd, blodtryck, kolesterolnivå, rökvanor och om de har en viss sjukdom.

Din uppgift är att analysera datan med Python och presentera resultaten i en Jupyter Notebook.

All kod, grafer och förklaringar ska finnas direkt i notebooken.

### Dataset

- Fil: `health_study_dataset.csv`
- `age` – ålder (år)
- `sex` – kön (M/F)
- `height` – längd (cm)
- `weight` – vikt (kg)
- `systolic_bp` – systoliskt blodtryck (mmHg)
- `cholesterol` – kolesterolnivå (mmol/L)
- `smoker` – rökvanor (Yes/No)
- `disease` – förekomst av sjukdom (0/1)

### Inlämning

Uppgiften lämnas in via GitHub i två delar:

- **Del 1 (vecka 45–46):** Grundläggande analys och statistik – branch `del1`
- **Del 2 (vecka 47–48):** Fördjupning och pipeline – branch `del2`

---

# Del 1 – Grundläggande analys och statistik

## För G ska du göra

### 1. Beskrivande analys

- Beräkna följande statistiska mått för variablerna:
  - `age`
  - `weight`
  - `height`
  - `systolic_bp`
  - `cholesterol`

- Måtten som ska beräknas är:
  - Medelvärde
  - Median
  - Minimum
  - Maximum

- Skapa minst **3 olika grafer**, exempelvis:
  - Histogram över blodtryck
  - Boxplot över vikt fördelat på kön
  - Stapeldiagram över andelen rökare

### 2. Simulering kopplad till caset

Använd `numpy.random` och en slump-seed.

- Beräkna andelen personer i datasetet som har sjukdomen
- Simulera 1 000 slumpmässiga personer med samma sannolikhet för sjukdom
- Jämför den simulerade andelen med den verkliga andelen i datasetet

### 3. Konfidensintervall

- Beräkna ett konfidensintervall för medelvärdet av `systolic_bp`
- Använd exempelvis normalapproximation eller bootstrap, som vi gått igenom på kursen

### 4. Hypotesprövning

Testa följande hypotes:

**"Rökare har högre medelblodtryck än icke-rökare."**

- Använd en av metoderna vi gått igenom, exempelvis bootstrap eller en t-test-funktion
- Förklara kort resultatet i en Markdown-cell

---

## För VG ska du även göra

### 5. Jämförelse av konfidensintervall

- Beräkna konfidensintervallet med **två olika metoder**, exempelvis normalapproximation och bootstrap
- Jämför resultaten och kommentera eventuella skillnader

### 6. Power – testets styrka

- Gör en simulering för att undersöka hur säkert ditt hypotesprov är
- Undersök exempelvis hur ofta testet hittar en statistisk skillnad när en sådan faktiskt finns (power)

### 7. Motivera metodval

- Motivera dina val av statistiska metoder i Markdown
- Hänvisa till dokumentation eller annan relevant källa

---

## Inlämning – Del 1

- Lämna in din Jupyter Notebook på GitHub
- Använd branchen `del1`

In [12]:
import pandas as pd         # Hanterar och analyserar tabeller (DataFrames)
import numpy as np          # Utför numeriska beräkningar och hanterar arrayer
import matplotlib.pyplot as plt # Skapar diagram och visualiseringar
from scipy import stats     # Hämtar statistiska funktioner från SciPy
import seaborn as snsb      # Skapar statistiska diagram
import math
import os                   # Hanterar bland annat filer och mappar
import sys                  # Ger tillgång till Pythons systeminställningar och sökvägar
sys.path.append("..")       # Lägger till mappen ovanför notebooks i Pythons importsökväg

df = pd.read_csv("../data/health_study_dataset.csv")

### Hur går simuleringen till?

Vi skapar nya, simulerade grupper med blodtrycksvärden:

- En grupp rökare
- En grupp icke-rökare

Vi bestämmer först hur stor verklig skillnad vi vill undersöka.

Exempelvis 2 mmHg. Det är ett antagande som vi väljer, inte ett resultat från vår data.

Sedan upprepar vi följande steg:

1. Simulera nya blodtrycksvärden för båda grupperna
2. Genomför samma typ av t-test som i vår hypotesprövning
3. Kontrollera om p-värdet är mindre än 0.05
4. Om p < 0.05 räknar vi simuleringen som en upptäckt
5. Upprepa hela processen exempelvis 1000 gånger

In [ ]:
%pip install statsmodels
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import NormalIndPower  # beräknar power (teststyrka)
from statsmodels.stats.proportion import proportion_effectsize  # beräknar effektstorlek mellan andelar
from statsmodels.stats.proportion import proportions_ztest  # utför z-test för att jämföra andelar
from statsmodels.stats.multitest import multipletests  # korrigerar p-värden vid flerfaldig testning
np.random.seed(42) # gör slumpresultaten reproducerbara

# ÖVERGRIPANDE UPPGIFT OCH SYFTE:
# Vi vill undersöka hur säkert vårt hypotesprov är när vi jämför blodtrycket
# mellan rökare (Y) och icke-rökare (N)
#
# Vi utgår från den observerade skillnaden mellan gruppernas medelvärden (0.4719 mmHg)
# och undersöker om vårt stickprov är tillräckligt stort för att upptäcka en sådan skillnad
#
# Vi beräknar först effektstorleken (Cohen's d) och därefter hur många personer
# som behövs i varje grupp för att uppnå 80 % power vid 5 % signifikansnivå
#
# Slutligen simulerar vi hypotesprovet för att undersöka hur ofta det upptäcker
# en verklig skillnad mellan grupperna

# ====================================================
# BASELINE
# ====================================================

smoker_yes = df.loc[df["smoker"] == "Yes", "systolic_bp"]
smoker_no = df.loc[df["smoker"] == "No", "systolic_bp"]

# Y = rökare (smoker_yes)
# N = icke-rökare (smoker_no)

baseline_mean_Y = 149.5249  # Medelvärdet för rökare
baseline_mean_N = 149.0530  # Medelvärdet för icke-rökare

# Skillnaden vi vill kunna upptäcka
min_detectable_diff = 0.4719

# Förväntat medelvärde för grupp N
# target_mean_N = medelvärdet för Y - minsta upptäckbara skillnaden
# 149.5249 - 0.4719 = 149.0530
target_mean_N = baseline_mean_Y - min_detectable_diff

alpha = 0.05  # Signifikansnivå: 5 %
power_target = 0.80  # Önskad power: 80 %

In [ ]:
# ==========================================================
# BERÄKNA EFFEKTSTORLEK (COHEN'S D) OCH NÖDVÄNDIG STICKPROVSSTORLEK
# ==========================================================

# 1. BERÄKNA EFFEKTSTORLEKEN (COHEN'S D)
# Cohen's D beräknas genom att dividera skillnaden mellan gruppernas medelvärden
# med gruppernas gemensamma standardavvikelse. Det Visar hur stor skillnaden är 
# i förhållande till spridningen

pooled_sd = np.sqrt((smoker_yes.std()**2 + smoker_no.std()**2) / 2) # gemensamma standardavvikelsen
effect_size = (baseline_mean_Y - baseline_mean_N) / pooled_sd

# SKRIVER UT EFFEKTSTORLEKEN
print(effect_size) # Cohen's d = 0.036

# TOLKNING:
# 0.036 är en mycket liten effektstorlek
# Skillnaden i genomsnittligt blodtryck mellan grupperna är liten
# jämfört med variationen i blodtryck mellan individerna inom grupperna

# vanliga riktmärken för Cohen's d:
# 0.2 = liten effekt
# 0.5 = medelstor effekt
# 0.8 = stor effekt

# Vi använder Cohen's d i vår poweranalys för att beräkna
# hur många personer som behövs för att upptäcka skillnaden
# En liten effektstorlek kräver generellt ett större stickprov

# 2: BERÄKNA ANTALET PERSONER SOM KRÄVS I VÅRT STICKPROV FÖR ATT UPPTÄCKA SKILLNADEN (0.4719)
# Syfte: Ta reda på hur mycket data som behövs för att inte bli lurad av slumpen

solver = NormalIndPower() # Skapar ett beräkningsverktyg (en "kalkylator") för test med två oberoende grupper
n_per_group = solver.solve_power(
    effect_size=effect_size, # Effektstorleken vi vill kunna upptäcka (Cohen's d = 0.036)
    alpha=alpha,             # Signifikansnivå: 0.05 (5 % risk att förkasta nollhypotesen trots att den stämmer)
    power=power_target,      # Träffsäkerhet: (oftast 0.8 = 80 %) används som en etablerad riktlinje inom statistik.
    ratio=1.0,              # Fördelning: 1.0 betyder exakt lika många personer i A som i B (50/50)
    alternative="larger"     # Ensidigt test: vi undersöker om rökare (Y) har högre blodtryck än icke-rökare (N)

)

n_per_group_rounded = int(np.ceil(n_per_group))

# 3: PRESENTERA RESULTATET
print(f"Exakt beräknat behov: {n_per_group}")
print(f"Antal personer som krävs per grupp: {n_per_group_rounded}")

0.03643763688702651
Exakt beräknat behov: 9313.174359378776
Antal personer som krävs per grupp: 9314


In [ ]:
# ==========================================================
# SKAPA EN SIMULERING FÖR ATT BERÄKNA STATISTISK STYRKA (POWER)
# ==========================================================

# SYFTE:
# Undersöka hur ofta vårt statistiska test lyckas upptäcka en verklig skillnad
# mellan rökare och icke-rökare (testets power)

# Vi simulerar hypotesprovet 5 000 gånger
# true_mean_Y och true_mean_N anger de antagna medelvärdena i grupperna
# alpha = 0.05 innebär 5 % signifikansnivå
# n_sims = 5000 anger hur många gånger vi upprepar simuleringen

def simulate_power(n_per_group, true_mean_Y, true_mean_N, alpha=0.05, n_sims=5000):

    rejections = 0  # Räknare för hur många gånger vi förkastar nollhypotesen (H0)

    for _ in range(n_sims):

        # Skapar slumpmässiga blodtrycksvärden för varje person i respektive grupp
        # Värdena dras från en normalfördelning med gruppens antagna medelvärde
        # och standardavvikelsen från vår ursprungliga data
        Y = np.random.normal(true_mean_Y, pooled_sd, n_per_group)  # Rökare
        N = np.random.normal(true_mean_N, pooled_sd, n_per_group)  # Icke-rökare

        # Utför ett ensidigt Welch t-test
        # Undersöker om rökare har högre genomsnittligt blodtryck än icke-rökare
        _, p_val = stats.ttest_ind(
            Y,
            N,
            equal_var=False,
            alternative="greater"
        )

        # Om p-värdet är mindre än alpha (0.05) förkastar vi H0
        # Vi räknar hur många simuleringar som hittar en statistiskt signifikant skillnad
        rejections += (p_val < alpha)

    # ==========================================================
    # BERÄKNA DEN SIMULERADE STATISTISKA STYRKAN (POWER)
    # ==========================================================

    # Andelen simuleringar där vi lyckades upptäcka skillnaden
    return rejections / n_sims


# KÖR SIMULERINGEN
sim_power = simulate_power(
    n_per_group_rounded,
    baseline_mean_Y,
    baseline_mean_N,
    alpha=alpha,
    n_sims=5000
)

print(f"Simulerad power: {sim_power:.3f}")